# Concorde TSP Solver

## Load costs matrix

In [221]:
import pandas as pd
import numpy as np
import math
import pulp
import heapq

df = pd.read_csv("bc_tsp_matrix.csv", index_col=0) #Load edges matrix

cities = df.index.tolist() #List of cities (names)

costs = df.values.tolist()

## Caro's LKH

In [222]:
# Caro's LKH implementation

def tour_cost(tour, costs):
    return sum(costs[tour[i]][tour[(i+1) % len(tour)]] for i in range(len(tour)))

def LKH(costs, initial_tour, max_iter=50, max_depth=3, dont_look_patience=2):
    n = len(costs)
    tour = initial_tour[:]
    best_tour = tour[:]
    best_cost = tour_cost(tour, costs)
    no_improve_count = 0

    dont_look = [0] * n

    def get_2opt_moves(current_tour):
        moves = []
        for i in range(n - 1):
            if dont_look[i] >= dont_look_patience:
                continue
            for j in range(i + 2, n):
                a, b = current_tour[i], current_tour[i + 1]
                c, d = current_tour[j], current_tour[(j + 1) % n]
                gain = (costs[a][b] + costs[c][d]) - (costs[a][c] + costs[b][d])
                if gain > 0:
                    moves.append((i, j, gain))
        return sorted(moves, key=lambda x: x[2], reverse=True)  # Mejores primero

    def apply_2opt(current_tour, i, j):
        new_tour = current_tour[:]
        new_tour[i + 1:j + 1] = reversed(new_tour[i +  1:j + 1])
        return new_tour

    def backtrack_search(current_tour, depth=0, accumulated_gain=0):
        if depth >= max_depth:
            return None, accumulated_gain

        moves = get_2opt_moves(current_tour)

        for i, j, gain in moves[:5]:
            new_tour = apply_2opt(current_tour, i, j)
            new_gain = accumulated_gain + gain

            result_tour, total_gain = backtrack_search(new_tour, depth + 1, new_gain)

            if result_tour is not None and total_gain > 0:
                return result_tour, total_gain

            if new_gain > 0:
                return new_tour, new_gain

        return None, accumulated_gain

    while no_improve_count < max_iter:
        improved_tour, total_gain = backtrack_search(tour)

        if improved_tour is not None:
            tour = improved_tour
            dont_look = [0] * n
            no_improve_count = 0

            current_cost = tour_cost(tour, costs)
            best_cost = current_cost
            best_tour = tour[:]
        else:
            dont_look = [d + 1 for d in dont_look]
            no_improve_count += 1

    return best_tour, best_cost


# Concorde Algorithm

In [223]:
# Function to initialize a problem using the pulp library
def initialize_lp(costs_matrix, fixed_edges, forbidden_edges, edges, n):
    problem = pulp.LpProblem("TSP", pulp.LpMinimize) # Create problem

    x = pulp.LpVariable.dicts("edge", edges, lowBound=0, upBound=1, cat='Continuous') # Create variables for edges

    problem += pulp.lpSum([costs_matrix[i][j] * x[(i,j)] for (i,j) in edges]) # Define objective function
    
    for i in range(n):
        connected_vars = [x[(u,v)] for (u,v) in edges if u == i or v == i] # Get the adjacent edges for a city
        problem += pulp.lpSum(connected_vars) == 2 # Degree constraint (the sum of the edges connected to a node must be 2)
    
    for (i,j) in fixed_edges:
        problem += x[(i,j)] == 1 # Force edge to be in the solution (branching)
    
    for (i,j) in forbidden_edges:
        problem += x[(i,j)] == 0 # Force edge to be excluded from the solution (branching)
    
    return problem, x

In [224]:
# Function to find subtours and apply subtour constraints (cutting step)
def add_subtour_cut(problem, x, n):
    # 1. Get edges with x > 0.5 (edges considered as active in the solution)
    active_edges = [(i, j) for (i, j), var in x.items() if var.value() is not None and var.value() > 0.5]

    # 2. Build list of adjacent edges for each node
    adj = {i: [] for i in range(n)}
    for i, j in active_edges:
        adj[i].append(j)
        adj[j].append(i)

    # 3. DFS from node 0
    visited = set()
    stack = [0]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        stack.extend(adj[node])

    # 4. Check for subtour
    if len(visited) == n:
        return False  # All nodes connected: No subtours

    subtour = list(visited)

    # 5. Add subtour elimination constraint (limit number of edges in a subtour to be <= to n_subtour-1 to avoid a cycle)
    problem += (pulp.lpSum( x[(min(i, j), max(i, j))] for i in subtour for j in subtour if i < j) <= len(subtour) - 1)

    return True

In [225]:
# Function to check if a solution is integer or not)
def check_integer_solution(x):
    for var in x.values():
        if var.value() is None or var.value() not in [0, 1]:
            return False # Variable different from 0 or 1 -> not integer solution
    return True

# Function to select an edge for branching (closest to 0.5)
def select_branch_edge(x):
    best_edge = None
    best_diff = float("inf")

    for (i, j), var in x.items():
        val = var.value()
        if val is None:
            continue
        if 1e-6 < val < 1 - 1e-6:  # fractional
            diff = abs(val - 0.5)
            if diff < best_diff:
                best_diff = diff
                best_edge = (i, j)

    return best_edge

In [226]:
def concorde_tsp(cities, costs_matrix):
    n = len(cities)
    edges  = [(i,j) for i in range(n) for j in range(n) if i < j] # Create list of edges as tuples

    # 1. Generate initial tour using LKH heuristic
    best_tour, initial_cost = LKH(costs_matrix, list(range(len(cities))), max_iter=10)

    # 2. Initialize LB and UP
    lower_bound = -math.inf
    global_upper_bound = initial_cost

    #global_upper_bound = math.inf # Test without LKH
    #best_tour = None # Test without LKH

    print(f"LKH initial solution (first global upper bound): {global_upper_bound}")

    # 3. Initialize LP relaxation 
    fixed_edges = []
    forbidden_edges = []
    queue = []
    
    # 4. Push initial problem node onto Priority Queue
    heapq.heappush(queue, (lower_bound, fixed_edges, forbidden_edges)) # Push initial problem to queue

    # 5. While Queue is not empty:
    while queue:
        current_lower_bound, fixed_edges, forbidden_edges = heapq.heappop(queue) # Pop node with lowest LB

        problem, x = initialize_lp(costs_matrix, fixed_edges, forbidden_edges, edges, n) # Initialize LP for current node

        if current_lower_bound >= global_upper_bound:
            continue # Prune if LB is worse than the global upper bound

        # Cutting loop
        while True:
            problem.solve()
            current_solution = problem.objective.value()

            if current_solution >= global_upper_bound:
                break # Prune if solution is worse than global upper bound

            # Subtour check and add constraints
            cut_added = add_subtour_cut(problem, x, n)

            if not cut_added:
                break  # LP is fully refined (no subtours)
        
        # Evaluation and branching
        if check_integer_solution(x):
            if current_solution < global_upper_bound:
                global_upper_bound = current_solution # Update upper bound
                best_tour = {index: var.varValue for index, var in x.items() if var.varValue == 1}
        else:
            edge_to_branch = select_branch_edge(x)
            heapq.heappush(queue, (current_solution, fixed_edges + [edge_to_branch], forbidden_edges)) # Branch on edge included
            heapq.heappush(queue, (current_solution, fixed_edges, forbidden_edges + [edge_to_branch])) # Branch on edge excluded

    # 6. Return Best Tour and UB
    return best_tour, global_upper_bound

## Test Algorithm

In [227]:
import time

start_time = time.time()

best_tour, best_cost = concorde_tsp(cities, costs)

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")

print(f"Solution: {best_cost} minutes = {best_cost / 60:.2f} hours")



LKH initial solution (first global upper bound): 916
Execution time: 0.025627851486206055 seconds
Solution: 916 minutes = 15.27 hours


In [228]:
from itertools import permutations

def brute_force_tsp(cities, costs_matrix):
    """
    Brute force solution for TSP.
    Tries all permutations of cities and returns the best tour.
    Time complexity: O(n!)
    Only practical for small instances (n < 12).
    """
    n = len(cities)
    
    # Start from city 0 to reduce permutations (10x speedup)
    if n <= 1:
        return list(range(n)), 0
    
    best_tour = None
    best_cost = float('inf')
    
    # Generate all permutations of remaining cities (fixing first city as 0)
    remaining_cities = list(range(1, n))
    
    for perm in permutations(remaining_cities):
        tour = [0] + list(perm)
        
        # Calculate tour cost
        cost = 0
        for i in range(n):
            cost += costs_matrix[tour[i]][tour[(i + 1) % n]]
        
        if cost < best_cost:
            best_cost = cost
            best_tour = tour
    
    return best_tour, best_cost


# Test brute force with the small problem
print("=== BRUTE FORCE TEST ===")
print(f"Number of cities: {len(cities)}")
brute_tour, brute_cost = brute_force_tsp(cities, costs)
print(f"Brute force best cost: {brute_cost}")
print(f"Brute force best tour: {brute_tour}")

# Compare with Concorde
print("\n=== CONCORDE TEST ===")
concorde_tour, concorde_cost = concorde_tsp(cities, costs)
print(f"Concorde best cost: {concorde_cost}")
print(f"Concorde best tour: {concorde_tour}")

# Comparison
print("\n=== COMPARISON ===")
print(f"Brute force cost: {brute_cost}")
print(f"Concorde cost: {concorde_cost}")
print(f"Difference: {abs(brute_cost - concorde_cost)}")
print(f"Concorde is optimal: {abs(brute_cost - concorde_cost) < 1e-6}")

=== BRUTE FORCE TEST ===
Number of cities: 7
Brute force best cost: 916
Brute force best tour: [0, 3, 1, 6, 5, 2, 4]

=== CONCORDE TEST ===
LKH initial solution (first global upper bound): 916
Concorde best cost: 916
Concorde best tour: [0, 3, 1, 6, 5, 2, 4]

=== COMPARISON ===
Brute force cost: 916
Concorde cost: 916
Difference: 0
Concorde is optimal: True


In [229]:
def tour_edges_to_sequence(best_tour, n):
    """
    Convert edge dictionary to tour sequence.
    best_tour is a dict of edges with value 1: {(i,j): 1}
    """
    if best_tour is None:
        return None
    
    # Handle case where best_tour is already a list
    if isinstance(best_tour, list):
        return best_tour
    
    # Build adjacency list from edges
    adj = {i: [] for i in range(n)}
    for (i, j) in best_tour.keys():
        adj[i].append(j)
        adj[j].append(i)
    
    # Reconstruct tour starting from city 0
    tour_sequence = [0]
    current = 0
    prev = None
    
    while len(tour_sequence) < n:
        # Find next neighbor (not the one we came from)
        for neighbor in adj[current]:
            if neighbor != prev:
                tour_sequence.append(neighbor)
                prev = current
                current = neighbor
                break
    
    return tour_sequence

# Convert Concorde tour to city names
if best_tour is not None:
    tour_sequence = tour_edges_to_sequence(best_tour, len(cities))
    tour_with_names = [cities[i] for i in tour_sequence]
    
    print("=== CONCORDE TOUR (City Names) ===")
    print("Tour order:")
    for idx, city in enumerate(tour_with_names):
        print(f"  {idx + 1}. {city}")
    
    # Print as a single line
    print("\nTour route:")
    print(" → ".join(tour_with_names) + " → " + tour_with_names[0])
    
    print(f"\nTotal cost: {concorde_cost}")
else:
    print("No tour found")

=== CONCORDE TOUR (City Names) ===
Tour order:
  1. Tijuana, Baja California, Mexico
  2. Tecate, Baja California, Mexico
  3. Mexicali, Baja California, Mexico
  4. San Felipe, Baja California, Mexico
  5. San Quintin, Baja California, Mexico
  6. Ensenada, Baja California, Mexico
  7. Rosarito, Baja California, Mexico

Tour route:
Tijuana, Baja California, Mexico → Tecate, Baja California, Mexico → Mexicali, Baja California, Mexico → San Felipe, Baja California, Mexico → San Quintin, Baja California, Mexico → Ensenada, Baja California, Mexico → Rosarito, Baja California, Mexico → Tijuana, Baja California, Mexico

Total cost: 916


In [230]:
def load_small_tsp():
    # 6 cities laid out roughly in a circle
    coords = [
        (0.0, 0.0),   # 0
        (1.0, 0.0),   # 1
        (2.0, 1.0),   # 2
        (1.0, 2.0),   # 3
        (0.0, 2.0),   # 4
        (-1.0, 1.0),  # 5
    ]
    return coords

import math

def build_distance_matrix(coords):
    n = len(coords)
    dist = [[0.0] * n for _ in range(n)]

    for i in range(n):
        x1, y1 = coords[i]
        for j in range(i + 1, n):
            x2, y2 = coords[j]
            d = math.hypot(x1 - x2, y1 - y2)
            dist[i][j] = d
            dist[j][i] = d

    return dist

coords = load_small_tsp()
costs_matrix = build_distance_matrix(coords)
cities = list(range(len(coords)))

# Test brute force with the small problem
print("=== BRUTE FORCE TEST ===")
print(f"Number of cities: {len(cities)}")
brute_tour, brute_cost = brute_force_tsp(cities, costs_matrix)
print(f"Brute force best cost: {brute_cost}")
print(f"Brute force best tour: {brute_tour}")

# Compare with Concorde
print("\n=== CONCORDE TEST ===")
concorde_tour, concorde_cost = concorde_tsp(cities, costs_matrix)
print(f"Concorde best cost: {concorde_cost}")
print(f"Concorde best tour: {concorde_tour}")

# Comparison
print("\n=== COMPARISON ===")
print(f"Brute force cost: {brute_cost}")
print(f"Concorde cost: {concorde_cost}")
print(f"Difference: {abs(brute_cost - concorde_cost)}")
print(f"Concorde is optimal: {abs(brute_cost - concorde_cost) < 1e-6}")

=== BRUTE FORCE TEST ===
Number of cities: 6
Brute force best cost: 7.65685424949238
Brute force best tour: [0, 1, 2, 3, 4, 5]

=== CONCORDE TEST ===
LKH initial solution (first global upper bound): 7.656854249492381
Concorde best cost: 7.65685424949238
Concorde best tour: {(0, 1): 1.0, (0, 5): 1.0, (1, 2): 1.0, (2, 3): 1.0, (3, 4): 1.0, (4, 5): 1.0}

=== COMPARISON ===
Brute force cost: 7.65685424949238
Concorde cost: 7.65685424949238
Difference: 0.0
Concorde is optimal: True
